# Q/A Dataset Curation

**Input**: `data/qa_dataset_raw.jsonl` (from Abdullah's `build_qa_dataset.py`)

**Output**:
- `data/qa_dataset.jsonl` — your curated keepers
- `data/qa_train.jsonl` — 80% train split
- `data/qa_eval.jsonl` — 20% eval split
- `data/curation_log.json` — your decisions (resume-safe)

## How to use
1. Run **Cell 1** (setup) once.
2. Run **Cell 2** (review loop) — for each example, click **Keep** or **Drop**.
3. Close the notebook anytime — your progress is saved. Re-run Cell 2 to resume.
4. When done, run **Cell 3** (split + save) to produce train/eval files.

## Decision guide
**KEEP** if: question is specific, answer is fully in the context, citation [1] makes sense.

**DROP** if: question is trivial, answer hallucinated (not in context), chunk is just math/references, or unanswerable without other chunks.

**Rule of thumb**: *Could a student answer this using ONLY the shown context? If no → drop.*

## Cell 1 — Setup (run once)

In [1]:
import json
import random
from pathlib import Path
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

# ---- Paths ----
DATA_DIR     = Path("../data")        # adjust if your notebook lives elsewhere
RAW_PATH     = DATA_DIR / "qa_dataset_raw.jsonl"
LOG_PATH     = DATA_DIR / "curation_log.json"
CURATED_PATH = DATA_DIR / "qa_dataset.jsonl"
TRAIN_PATH   = DATA_DIR / "qa_train.jsonl"
EVAL_PATH    = DATA_DIR / "qa_eval.jsonl"

# ---- Load raw data ----
assert RAW_PATH.exists(), f"Missing file: {RAW_PATH.resolve()}"
raw = []
with RAW_PATH.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            print(f"Skipping bad JSON on line {i+1}")
            continue
        # Skip null/empty examples that the generator marked unusable
        if not obj or not obj.get("question") or not obj.get("answer"):
            continue
        obj["_idx"] = i
        raw.append(obj)

# ---- Load (or initialize) log ----
if LOG_PATH.exists():
    log = json.loads(LOG_PATH.read_text(encoding="utf-8"))
else:
    log = {"decisions": {}}  # { idx_str: "keep" | "drop" }

def save_log():
    LOG_PATH.write_text(json.dumps(log, indent=2), encoding="utf-8")

decided = set(int(k) for k in log["decisions"].keys())
pending = [r for r in raw if r["_idx"] not in decided]

kept    = sum(1 for v in log["decisions"].values() if v == "keep")
dropped = sum(1 for v in log["decisions"].values() if v == "drop")

print(f"Loaded {len(raw)} raw examples.")
print(f"Already decided: {len(decided)}  (keep={kept}, drop={dropped})")
print(f"Pending review: {len(pending)}")

Loaded 276 raw examples.
Already decided: 0  (keep=0, drop=0)
Pending review: 276


## Cell 2 — Review loop

Click **Keep** or **Drop**. The next example loads automatically. **Undo** reverts the last decision. **Save & Exit** writes progress and stops.

Your progress is saved after every click, so closing the notebook mid-way is safe.

In [2]:
state = {"queue": list(pending), "history": [], "stopped": False}

out = widgets.Output()

btn_keep   = widgets.Button(description="✅ Keep",   button_style="success", layout=widgets.Layout(width="120px"))
btn_drop   = widgets.Button(description="❌ Drop",   button_style="danger",  layout=widgets.Layout(width="120px"))
btn_undo   = widgets.Button(description="↩ Undo",    button_style="warning", layout=widgets.Layout(width="120px"))
btn_exit   = widgets.Button(description="💾 Save & Exit", button_style="info", layout=widgets.Layout(width="160px"))

def render_current():
    with out:
        clear_output(wait=True)
        kept_now    = sum(1 for v in log["decisions"].values() if v == "keep")
        dropped_now = sum(1 for v in log["decisions"].values() if v == "drop")
        total_done  = kept_now + dropped_now
        total_all   = len(raw)

        if state["stopped"]:
            display(HTML(f"<h3>💾 Saved and stopped.</h3>"
                         f"<p>Progress: <b>{total_done}/{total_all}</b> "
                         f"(keep={kept_now}, drop={dropped_now}).</p>"
                         f"<p>Re-run this cell anytime to resume.</p>"))
            return

        if not state["queue"]:
            display(HTML(f"<h3>🎉 All done!</h3>"
                         f"<p>Final: <b>keep={kept_now}, drop={dropped_now}</b> out of {total_all}.</p>"
                         f"<p>Now run <b>Cell 3</b> to produce the train/eval splits.</p>"))
            return

        item = state["queue"][0]
        ctx  = item.get("context", "")
        # Trim very long contexts for display only (data is unchanged)
        ctx_display = ctx if len(ctx) < 1500 else ctx[:1500] + " …[truncated for display]"

        html = f"""
        <div style="font-family: -apple-system, system-ui, sans-serif; max-width: 900px;">
          <div style="color:#666; font-size: 13px; margin-bottom: 8px;">
            Progress: <b>{total_done}/{total_all}</b>
            &nbsp;·&nbsp; keep={kept_now} &nbsp;·&nbsp; drop={dropped_now}
            &nbsp;·&nbsp; remaining: {len(state['queue'])}
          </div>
          <div style="color:#999; font-size: 12px;">
            paper_id: <code>{item.get('paper_id','?')}</code>
            &nbsp;·&nbsp; chunk_id: <code>{item.get('chunk_id','?')}</code>
            &nbsp;·&nbsp; page: <code>{item.get('page','?')}</code>
          </div>
          <hr/>
          <div style="background:#f0f7ff; padding:12px; border-left:4px solid #2563eb; margin:8px 0;">
            <b>QUESTION</b><br/>
            <div style="font-size:15px; margin-top:4px;">{item.get('question','')}</div>
          </div>
          <div style="background:#fafafa; padding:12px; border-left:4px solid #888; margin:8px 0; white-space:pre-wrap;">
            <b>CONTEXT (the chunk)</b><br/>
            <div style="font-size:13px; margin-top:4px; line-height:1.5;">{ctx_display}</div>
          </div>
          <div style="background:#f0fff4; padding:12px; border-left:4px solid #16a34a; margin:8px 0;">
            <b>PROPOSED ANSWER</b><br/>
            <div style="font-size:14px; margin-top:4px;">{item.get('answer','')}</div>
          </div>
          <div style="color:#666; font-size:12px; margin-top:6px;">
            ✅ Keep if: question is specific · answer is fully in context · [1] is correct.<br/>
            ❌ Drop if: trivial · hallucinated · chunk is math/refs only · needs other chunks.
          </div>
        </div>
        """
        display(HTML(html))

def decide(label):
    if state["stopped"] or not state["queue"]:
        return
    item = state["queue"].pop(0)
    log["decisions"][str(item["_idx"])] = label
    state["history"].append(item)
    save_log()
    render_current()

def on_keep(b): decide("keep")
def on_drop(b): decide("drop")

def on_undo(b):
    if not state["history"]:
        return
    last = state["history"].pop()
    log["decisions"].pop(str(last["_idx"]), None)
    state["queue"].insert(0, last)
    save_log()
    render_current()

def on_exit(b):
    state["stopped"] = True
    save_log()
    render_current()

btn_keep.on_click(on_keep)
btn_drop.on_click(on_drop)
btn_undo.on_click(on_undo)
btn_exit.on_click(on_exit)

controls = widgets.HBox([btn_keep, btn_drop, btn_undo, btn_exit])
display(controls, out)
render_current()

Output()

## Cell 3 — Build train/eval splits

Run this **after** you've reviewed everything (or enough). It writes:
- `data/qa_dataset.jsonl` — all kept examples
- `data/qa_train.jsonl` — 80%
- `data/qa_eval.jsonl` — 20%

The split is deterministic (seed=42) so the same examples land in train/eval every time.

In [3]:
SEED       = 42
EVAL_RATIO = 0.20

# Reload decisions (in case Cell 2 was re-run)
log = json.loads(LOG_PATH.read_text(encoding="utf-8"))
kept_idxs = {int(k) for k, v in log["decisions"].items() if v == "keep"}

kept_examples = []
for r in raw:
    if r["_idx"] in kept_idxs:
        clean = {k: v for k, v in r.items() if k != "_idx"}
        kept_examples.append(clean)

print(f"Total kept: {len(kept_examples)}")

if len(kept_examples) < 50:
    print("⚠️  Fewer than 50 kept examples. Consider reviewing more before training.")

# Deterministic shuffle
rng = random.Random(SEED)
shuffled = kept_examples[:]
rng.shuffle(shuffled)

n_eval = max(1, int(round(len(shuffled) * EVAL_RATIO)))
eval_set  = shuffled[:n_eval]
train_set = shuffled[n_eval:]

def write_jsonl(path, rows):
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

write_jsonl(CURATED_PATH, kept_examples)
write_jsonl(TRAIN_PATH,   train_set)
write_jsonl(EVAL_PATH,    eval_set)

print(f"\n✅ Wrote:")
print(f"  {CURATED_PATH}  ({len(kept_examples)} examples)")
print(f"  {TRAIN_PATH}    ({len(train_set)} examples, 80%)")
print(f"  {EVAL_PATH}     ({len(eval_set)} examples, 20%)")

# Sanity check — no paper_id leakage where possible (warn only)
train_papers = {r.get("paper_id") for r in train_set}
eval_papers  = {r.get("paper_id") for r in eval_set}
overlap = train_papers & eval_papers
print(f"\nPaper overlap between train/eval: {len(overlap)} papers")
print("(Some overlap is fine — we're testing generalization within the corpus, not across new papers.)")

Total kept: 218

✅ Wrote:
  ..\data\qa_dataset.jsonl  (218 examples)
  ..\data\qa_train.jsonl    (174 examples, 80%)
  ..\data\qa_eval.jsonl     (44 examples, 20%)

Paper overlap between train/eval: 29 papers
(Some overlap is fine — we're testing generalization within the corpus, not across new papers.)


## Done?

Commit the result:

```bash
git add data/qa_dataset.jsonl data/qa_train.jsonl data/qa_eval.jsonl data/curation_log.json
git commit -m "data(d4): curated Q/A dataset + train/eval split"
git push origin d4-dev
```

Then ping Abdullah — he can now run `scripts/train_qlora.py` on `data/qa_train.jsonl`.